# Landing → Bronze: ocorrencias_rede

**Fonte:** `ocorrencias_rede.csv` (interrupções de fornecimento; cada registro é uma UC interrompida), separador vírgula, com cabeçalho.
**Destino:** `venergy.bronze.ocorrencias_rede` (Delta).

**Decisões desta camada**
- Conteúdo preservado: schema explícito com todas as colunas como texto (`STRING`), equivalente a `inferSchema = false`. Sem `trim`, limpeza, deduplicação ou conversão. O tratamento é feito na Silver.
- Contrato do cabeçalho: com `enforceSchema = false`, o Spark compara o cabeçalho do arquivo com `EXPECTED_COLUMNS`. Se a origem renomear ou reordenar colunas, a carga falha em vez de gravar dados deslocados.
- Linhas malformadas: `mode = FAILFAST`. Uma linha com número de colunas diferente do contrato interrompe a carga em vez de ser descartada ou completada com `null`.
- Colunas de rastreio: `source_file` (arquivo de origem) e `ingestion_datetime` (momento da carga), padrão do time.
- Carga completa com `overwrite`: o dataset é uma extração única, então reexecutar o notebook não duplica registros (o Workflow precisa ser reexecutável de ponta a ponta).
- Validação antes da escrita: a contagem do arquivo (lido como texto puro) e a do DataFrame são conferidas com `EXPECTED_ROWS` **antes** do `overwrite`. Se divergirem, a Bronze atual não é tocada. Cabeçalho fora do contrato e linha malformada também interrompem a carga sem publicar nada, porque a escrita Delta só é confirmada no fim. Depois de gravar, a tabela é conferida mais uma vez.
- Observação: o leitor CSV do Spark transforma campo vazio em `null`. Os demais marcadores de ausência em `causa` (`-`, `N/A`, `NULL`) chegam como texto, e `duracao_min` mantém o sufixo `" min"` quando ele existe no bruto.

As falhas encontradas estão em `docs/auditoria_dados/auditoria_ocorrencias_rede.md`.

In [ ]:
catalog = "venergy"
landing_schema = "landing"
bronze_schema = "bronze"
landing_volume = "landing_volume"
volume_path = f"/Volumes/{catalog}/{landing_schema}/{landing_volume}"

ocorrencias_path = f"{volume_path}/dados/ocorrencias_rede.csv"
bronze_table = f"{catalog}.{bronze_schema}.ocorrencias_rede"

EXPECTED_COLUMNS = [
    "id_ocorrencia",
    "alimentador",
    "id_uc",
    "tipo",
    "causa",
    "data_abertura",
    "data_encerramento",
    "duracao_min",
]
EXPECTED_ROWS = 180_000

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{landing_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{landing_schema}.{landing_volume}")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

schema_bruto = StructType([StructField(c, StringType(), True) for c in EXPECTED_COLUMNS])

df_ocorrencias = (
    spark.read
    .schema(schema_bruto)
    .option("header", True)
    .option("enforceSchema", False)
    .option("mode", "FAILFAST")
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .option("ignoreLeadingWhiteSpace", False)
    .option("ignoreTrailingWhiteSpace", False)
    .csv(ocorrencias_path)
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("ingestion_datetime", F.current_timestamp())
)

In [ ]:
# Antes de gravar: contagem do arquivo lido como texto puro (sem parser CSV), descontando o cabeçalho,
# comparada com o DataFrame lido pelo contrato e com o volume esperado da extração.
# Se algo divergir, o assert interrompe aqui e a Bronze atual não é tocada.
linhas_arquivo = spark.read.text(ocorrencias_path).filter(F.length("value") > 0).count() - 1
linhas_df = df_ocorrencias.count()

print({"linhas_arquivo": linhas_arquivo, "linhas_df": linhas_df, "esperado": EXPECTED_ROWS})
assert linhas_df == linhas_arquivo, "Contagem do DataFrame diverge do arquivo bruto"
assert linhas_df == EXPECTED_ROWS, "Contagem diverge do volume esperado da extração"

In [ ]:
(
    df_ocorrencias.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

In [ ]:
# Depois de gravar: a tabela tem exatamente as colunas do contrato + rastreio e o volume esperado
tabela = spark.table(bronze_table)
linhas_tabela = tabela.count()

assert tabela.columns == EXPECTED_COLUMNS + ["source_file", "ingestion_datetime"], f"Colunas inesperadas: {tabela.columns}"
assert linhas_tabela == EXPECTED_ROWS, f"Tabela com {linhas_tabela} linhas, esperado {EXPECTED_ROWS}"
print("OK: 180.000 linhas carregadas em", bronze_table)

## Evidências das falhas (consultas sobre a Bronze)
As consultas abaixo reproduzem os principais achados do relatório de auditoria. Elas não alteram dados.

In [ ]:
t = bronze_table
duracao = "try_cast(regexp_replace(duracao_min, ' min$', '') AS INT)"
duracao_datas = "(unix_timestamp(data_encerramento) - unix_timestamp(data_abertura)) / 60"

# 1. Valores de causa, incluindo os marcadores de ausência
display(spark.sql(f"""
  SELECT coalesce(causa, '<null>') AS causa, COUNT(*) AS linhas
  FROM {t} GROUP BY 1 ORDER BY 2 DESC"""))

# 2. duracao_min: sufixo " min" e coerência com as datas
display(spark.sql(f"""
  SELECT COUNT_IF(duracao_min LIKE '% min') AS com_sufixo_min,
         COUNT_IF({duracao} IS NULL) AS nao_numericos_sem_sufixo,
         COUNT_IF({duracao} <> {duracao_datas}) AS divergentes_das_datas,
         COUNT_IF(data_encerramento < data_abertura) AS encerramento_antes_da_abertura,
         COUNT_IF({duracao} >= 1440) AS duracao_24h_ou_mais,
         MAX({duracao}) AS duracao_maxima_min
  FROM {t}"""))

# 3. Ocorrências por mês de abertura (pico em 2025-07)
display(spark.sql(f"""
  SELECT substr(data_abertura, 1, 7) AS mes, COUNT(*) AS ocorrencias
  FROM {t} GROUP BY 1 ORDER BY 1"""))

# 4. Concentração do pico de 2025-07 por alimentador e causa
display(spark.sql(f"""
  SELECT alimentador, coalesce(causa, '<null>') AS causa, COUNT(*) AS ocorrencias
  FROM {t} WHERE data_abertura LIKE '2025-07%'
  GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 10"""))

# 5. Interrupções sobrepostas na mesma UC: abertura antes do MAIOR encerramento anterior da UC
#    (não só da ocorrência imediatamente anterior: uma interrupção longa pode conter uma curta e uma terceira)
fim_anterior = """MAX(data_encerramento) OVER (
    PARTITION BY id_uc ORDER BY data_abertura, data_encerramento
    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)"""
display(spark.sql(f"""
  SELECT substr(data_abertura, 1, 7) AS mes, COUNT(*) AS sobrepostas
  FROM (SELECT data_abertura, {fim_anterior} AS maior_fim_anterior FROM {t})
  WHERE data_abertura < maior_fim_anterior
  GROUP BY 1 ORDER BY 2 DESC"""))

# 6. Minutos contados em dobro: soma direta das durações x união dos intervalos de cada UC
display(spark.sql(f"""
  WITH ordenado AS (
    SELECT id_uc, data_abertura, data_encerramento, {fim_anterior} AS maior_fim_anterior FROM {t}
  ), ilhas AS (
    SELECT *, SUM(CASE WHEN maior_fim_anterior IS NULL OR data_abertura >= maior_fim_anterior THEN 1 ELSE 0 END)
                OVER (PARTITION BY id_uc ORDER BY data_abertura, data_encerramento ROWS UNBOUNDED PRECEDING) AS ilha
    FROM ordenado
  ), unidas AS (
    SELECT id_uc, ilha, MIN(data_abertura) AS inicio, MAX(data_encerramento) AS fim FROM ilhas GROUP BY id_uc, ilha
  ), totais AS (
    SELECT (SELECT SUM(unix_timestamp(data_encerramento) - unix_timestamp(data_abertura)) / 60 FROM ordenado) AS somados,
           (SELECT SUM(unix_timestamp(fim) - unix_timestamp(inicio)) / 60 FROM unidas) AS sem_energia
  )
  SELECT CAST(somados AS BIGINT) AS minutos_somados,
         CAST(sem_energia AS BIGINT) AS minutos_sem_energia_unidos,
         CAST(somados - sem_energia AS BIGINT) AS minutos_em_dobro
  FROM totais"""))

# 7. Cobertura: período, alimentadores, municípios (prefixo do alimentador) e UCs
display(spark.sql(f"""
  SELECT MIN(data_abertura) AS primeira_abertura, MAX(data_abertura) AS ultima_abertura,
         COUNT(DISTINCT alimentador) AS alimentadores,
         COUNT(DISTINCT substr(alimentador, 5, 3)) AS prefixos_municipio,
         COUNT(DISTINCT id_uc) AS ucs_afetadas,
         COUNT(DISTINCT id_ocorrencia) AS ids_distintos
  FROM {t}"""))